# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema accessible via URL and contains several record sets and fields referencing all entities by their `@id`, in line with best practices for interoperable data.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

**Note:** All dataset entities (record sets, fields, columns) are referenced by their `@id`, following best practices.

In [ ]:
# List all record sets and their fields
print('Available Record Sets:')

record_sets = []
for record_set in dataset.record_sets:
    print(f"- RecordSet name: {record_set.name} | @id: {record_set.identifier}")
    record_sets.append(record_set.identifier)

    print("  Fields (@id):")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.identifier}) | dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis, using the `@id` values.

You may select specific record sets or extract all for comprehensive exploration.

In [ ]:
# Prepare DataFrames for all record sets
dataframes = {}

for record_set_id in record_sets:
    # Using the @id for each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:  # If the record set has records
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}, {len(df)} rows, columns: {list(df.columns)}\n")

# Example: Show the first few rows of the first available record set
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Example preview for record set: {first_rs}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering based on field values, normalizing numeric attributes, removing outliers, or grouping by demographic variables.

For demonstration, let's select fields by their `@id` (for instance, 'gad7_total' for GAD-7 score and 'age' for age). You should replace these with the correct field `@id` from step 2 above.

In [ ]:
# Example (Replace with actual field @id from your record set overview):
# For demonstration, assume first_rs contains 'gad7_total' (GAD-7 score) and 'age' fields.
record_set_id = first_rs  # Use the record set for main survey data
df = dataframes[record_set_id]

# You must set the correct @id for the desired fields (from previous overview output)
numeric_field_id = 'gad7_total'  # Example field @id for GAD-7 total score
group_field_id = 'gender'        # Example field @id for Gender

# Proceed only if these fields exist (they will, if present in the dataset)
if numeric_field_id in df.columns:
    threshold = 5
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean GAD-7 scores by {group_field_id}:")
        display(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize distributions or relationships in the dataset using pandas and matplotlib. Below, we plot a histogram of the GAD-7 score and a boxplot comparison by gender, referencing the correct field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Plot histogram of GAD-7 scores
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True, color='skyblue')
    plt.title('Distribution of GAD-7 Total Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

# Boxplot of GAD-7 by Gender
if group_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id], palette='pastel')
    plt.title('GAD-7 Score by Gender')
    plt.xlabel('Gender')
    plt.ylabel('GAD-7 Score')
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library. By referencing all entities by their `@id`, we've ensured transparent and unambiguous analysis. From the summary and EDA, you can proceed to domain-specific analysis, build models, or further visualize key trends in the data.